In [1]:
import pandas as pd

# 파일 경로 (사용자님 환경에 맞게 수정)
file_path = "S-DoT_NATURE_2022/S-DoT_NATURE_2022.01.03-01.09.csv"
df = pd.read_csv(file_path, encoding="cp949")

# 데이터의 크기와 널값 개수 확인
print(f"전체 데이터 개수: {len(df)}")
print(df[['소음(dB)', '진동_x(g)']].isnull().sum()) # 널값이 얼마나 있는지 확인

전체 데이터 개수: 168989
소음(dB)       795
진동_x(g)    30410
dtype: int64


In [2]:
# 필요한 컬럼만 추출
target_cols = ['시리얼', '소음(dB)', '진동_x(g)', '진동_y(g)', '진동_z(g)']
df_selected = df[target_cols].copy()

# 시리얼별 평균 계산 (NaN은 자동으로 제외하고 계산됨)
weekly_avg = df_selected.groupby('시리얼').mean().reset_index()

# 결과 확인
print(weekly_avg.head())

           시리얼     소음(dB)  진동_x(g)  진동_y(g)  진동_z(g)
0  OC3CL200011  51.273885      NaN      NaN      NaN
1  OC3CL200012  58.121019      NaN      NaN      NaN
2  OC3CL200013  47.791139      NaN      NaN      NaN
3  OC3CL200014  58.044586      NaN      NaN      NaN
4  OC3CL200016  49.063291      NaN      NaN      NaN


In [3]:
# 1. 시리얼별 평균 계산
# (사용자님이 진행하신 코드에 '소음'에 집중해서 결과를 보겠습니다)
weekly_avg = df_selected.groupby('시리얼').mean().reset_index()

# 2. 소음 기준 내림차순 정렬 (가장 시끄러운 곳부터 확인)
weekly_avg_sorted = weekly_avg.sort_values(by='소음(dB)', ascending=False)

# 3. 결과 출력
print("--- 주간 평균 소음 상위 20개 지점 ---")
print(weekly_avg_sorted[['시리얼', '소음(dB)']].head(20))

print("\n--- 주간 평균 소음 하위 10개 지점 ---")
print(weekly_avg_sorted[['시리얼', '소음(dB)']].tail(10))

--- 주간 평균 소음 상위 20개 지점 ---
             시리얼     소음(dB)
235  OC3CL200254  71.045455
110  OC3CL200123  70.577922
224  OC3CL200243  70.445161
202  OC3CL200215  70.444444
151  OC3CL200164  69.871795
161  OC3CL200174  69.842767
62   OC3CL200075  69.753247
114  OC3CL200127  69.688312
157  OC3CL200170  69.544872
206  OC3CL200219  69.529412
142  OC3CL200155  69.272152
204  OC3CL200217  69.254902
22   OC3CL200035  69.123377
192  OC3CL200205  68.968153
207  OC3CL200220  68.849673
129  OC3CL200142  68.516129
188  OC3CL200201  68.337580
8    OC3CL200020  68.075949
28   OC3CL200041  67.993464
113  OC3CL200126  67.837662

--- 주간 평균 소음 하위 10개 지점 ---
              시리얼     소음(dB)
352   V02Q1940158  35.065359
143   OC3CL200156  35.044304
330   V02Q1940136  35.019355
38    OC3CL200051  35.006623
765   V02Q1940585  35.000000
1069  V02Q1940935        NaN
1085  V02Q1941000        NaN
1086  V02Q1941006        NaN
1087  V02Q1941013        NaN
1088  V02Q1941015        NaN


In [4]:
import pandas as pd

# 1. 위치 정보 파일 불러오기
df_loc = pd.read_csv('S-DoT_NATURE_2022/serial_gu.csv', encoding='utf-8')

# 2. 안전하게 자치구 추출하는 함수 정의
def extract_gu(address):
    if pd.isnull(address):
        return "주소없음"
    
    parts = str(address).split()
    
    # 단어가 2개 이상일 때만 두 번째 단어(구)를 가져옴
    if len(parts) >= 2:
        return parts[1]
    else:
        # "서울대공원" 같은 경우 처리 (보통 과천에 있지만 행정구역상 서울 소속일 수 있음)
        # 상황에 따라 "기타" 혹은 첫 번째 단어를 반환
        return "기타(단일주소)"

# 3. 함수 적용
df_loc['자치구'] = df_loc['주소'].apply(extract_gu)

# 4. 결과 확인 (잘 나뉘었는지 체크)
print(df_loc[['주소', '자치구']].iloc[1060:1065]) # 아까 에러 났던 구간 확인

                        주소       자치구
1060                 서울대공원  기타(단일주소)
1061                 서울대공원  기타(단일주소)
1062                 서울대공원  기타(단일주소)
1063                 서울대공원  기타(단일주소)
1064  서울특별시 용산구 한강대로38길 35       용산구


In [5]:
import pandas as pd

# 1. 위치 정보 전처리 (시리얼, 자치구, 위도, 경도 추출)
# df_loc는 이미 이전 단계에서 불러와 '자치구' 컬럼을 만든 상태여야 합니다.
# 파일의 실제 컬럼명인 '모델 시리얼(*)'을 '시리얼'로 변경하여 매핑 준비를 합니다.
df_loc_map = df_loc[['모델 시리얼(*)', '자치구', '위도', '경도']].copy()
df_loc_map.columns = ['시리얼', '자치구', '위도', '경도']

# 2. 시리얼별 평균 계산 (기존에 작성하신 코드)
# df_selected 데이터에서 시리얼별로 평균을 냅니다.
weekly_avg = df_selected.groupby('시리얼').mean(numeric_only=True).reset_index()

# 3. 데이터 병합 (Merge)
# 소음 데이터(weekly_avg)에 위치 정보(df_loc_map)를 시리얼 기준으로 붙입니다.
final_df = pd.merge(weekly_avg, df_loc_map, on='시리얼', how='left')

# 4. 소음 기준 내림차순 정렬
final_sorted = final_df.sort_values(by='소음(dB)', ascending=False)

# 5. 결과 출력 (상위 20개 지점)
print("--- 주간 평균 소음 상위 20개 지점 (자치구 및 좌표 포함) ---")
# 필요한 컬럼만 순서대로 출력
print(final_sorted[['시리얼', '자치구', '소음(dB)']].head(20))

# 6. 결과 출력 (하위 10개 지점)
print("\n--- 주간 평균 소음 하위 10개 지점 ---")
print(final_sorted[['시리얼', '자치구', '소음(dB)']].tail(10))

--- 주간 평균 소음 상위 20개 지점 (자치구 및 좌표 포함) ---
             시리얼   자치구     소음(dB)
235  OC3CL200254   마포구  71.045455
110  OC3CL200123   강동구  70.577922
224  OC3CL200243   마포구  70.445161
202  OC3CL200215  서대문구  70.444444
151  OC3CL200164   성동구  69.871795
161  OC3CL200174  영등포구  69.842767
62   OC3CL200075   서초구  69.753247
114  OC3CL200127    중구  69.688312
157  OC3CL200170   강남구  69.544872
206  OC3CL200219   종로구  69.529412
142  OC3CL200155   강남구  69.272152
204  OC3CL200217   성동구  69.254902
22   OC3CL200035   은평구  69.123377
192  OC3CL200205   강동구  68.968153
207  OC3CL200220   종로구  68.849673
129  OC3CL200142   광진구  68.516129
188  OC3CL200201   종로구  68.337580
8    OC3CL200020   강남구  68.075949
28   OC3CL200041    중구  67.993464
113  OC3CL200126    중구  67.837662

--- 주간 평균 소음 하위 10개 지점 ---
              시리얼   자치구     소음(dB)
352   V02Q1940158   서초구  35.065359
143   OC3CL200156   강남구  35.044304
330   V02Q1940136   구로구  35.019355
38    OC3CL200051   도봉구  35.006623
765   V02Q1940585  서대문구  35.000000
1069  V

In [6]:
import pandas as pd

# 1. 2023년 파일에서 '시리얼-지역' 매핑 테이블 만들기
df_2023 = pd.read_csv('S-DoT_NATURE_2022/S-DoT_NATURE_2023.01.16-01.22.csv', encoding='cp949')
# 시리얼별로 지역 정보는 동일하므로 중복을 제거하여 매핑용으로 사용
region_map = df_2023[['시리얼', '지역']].drop_duplicates('시리얼')

# 2. 2022년 환경 데이터 로드 및 평균 계산
df_2022 = pd.read_csv('S-DoT_NATURE_2022/S-DoT_NATURE_2022.01.03-01.09.csv', encoding='cp949')
target_cols_2022 = ['시리얼', '소음(dB)', '진동_x(g)', '진동_y(g)', '진동_z(g)']
# 2022년 데이터의 주간 평균 산출
weekly_avg_2022 = df_2022[target_cols_2022].groupby('시리얼').mean(numeric_only=True).reset_index()

# 3. [핵심] 2022년 평균 데이터에 2023년의 지역 정보를 '시리얼' 기준으로 합치기
# 2022년 데이터(Left)를 기준으로 2023년의 지역 정보(Right)를 붙입니다.
final_2022_with_region = pd.merge(weekly_avg_2022, region_map, on='시리얼', how='left')

# 4. 위도/경도 정보까지 추가하고 싶다면 (선택사항)
# 아까 만드신 df_loc_map(시리얼, 위도, 경도)을 한 번 더 합칩니다.
final_result = pd.merge(final_2022_with_region, df_loc_map[['시리얼', '자치구']], on='시리얼', how='left')

# 5. 결과 확인
print("--- 2023년 지역 정보가 매핑된 2022년 주간 평균 ---")
print(final_result.head())

--- 2023년 지역 정보가 매핑된 2022년 주간 평균 ---
           시리얼     소음(dB)  진동_x(g)  진동_y(g)  진동_z(g)                   지역  \
0  OC3CL200011  51.273885      NaN      NaN      NaN                parks   
1  OC3CL200012  58.121019      NaN      NaN      NaN  traditional_markets   
2  OC3CL200013  47.791139      NaN      NaN      NaN          main_street   
3  OC3CL200014  58.044586      NaN      NaN      NaN          main_street   
4  OC3CL200016  49.063291      NaN      NaN      NaN          main_street   

        자치구  
0  기타(단일주소)  
1       광진구  
2       종로구  
3        중구  
4       강남구  


In [7]:
# 1. 자치구별로 그룹화하여 평균 계산
# 시리얼 번호는 수치 데이터가 아니거나 분석에서 제외하므로 numeric_only=True를 사용합니다.
gu_avg = final_result.groupby('자치구')[['소음(dB)', '진동_x(g)', '진동_y(g)', '진동_z(g)']].mean().reset_index()

# 2. 소음 수치가 높은 순서대로 정렬 (가장 시끄러운 자치구 순)
gu_avg_sorted = gu_avg.sort_values(by='소음(dB)', ascending=False)

# 3. 결과 출력
print("--- 자치구별 주간 평균 환경 지표 ---")
print(gu_avg_sorted)

--- 자치구별 주간 평균 환경 지표 ---
         자치구     소음(dB)   진동_x(g)   진동_y(g)   진동_z(g)
1        강동구  53.514216  0.031403  0.066547  1.038275
15       서초구  51.930951  0.025635  0.065486  1.042878
0        강남구  51.876400  0.031071  0.069410  1.036234
3        강서구  51.466982  0.026829  0.050815  1.045919
8   기타(단일주소)  51.399029       NaN       NaN       NaN
22       은평구  50.145728  0.028336  0.059949  1.052154
24        중구  50.127818  0.023512  0.061626  1.039502
14      서대문구  49.623645  0.036502  0.060599  1.039034
5        광진구  49.518005  0.029559  0.059203  1.042110
12       동작구  49.458582  0.022117  0.066828  1.036355
13       마포구  49.384898  0.031706  0.064209  1.043191
10       도봉구  49.045199  0.025173  0.072216  1.038502
2        강북구  48.850298  0.026749  0.054834  1.041717
23       종로구  48.815233  0.032561  0.048350  1.043568
6        구로구  48.765770  0.029871  0.063596  1.050199
20      영등포구  48.665073  0.024113  0.051432  1.050215
9        노원구  48.646849  0.028625  0.058502  1.031338
7  

In [8]:
# 1. '기타(단일주소)' 및 '주소없음' 제외 (필터링)
df_filtered = final_result[~final_result['자치구'].isin(['기타(단일주소)', '주소없음'])].copy()

# 2. [소음] 자치구별 & 지역별 평균 계산
# 자치구 내에서도 어느 동네(지역)가 시끄러운지 세부적으로 확인 가능합니다.
noise_by_gu_region = df_filtered.groupby(['자치구', '지역'])['소음(dB)'].mean().reset_index()
noise_by_gu_region = noise_by_gu_region.sort_values(by=['자치구', '소음(dB)'], ascending=[True, False])

# 3. [진동] 자치구별 평균 계산 (시리얼, 지역 정보 제외)
vibration_by_gu = df_filtered.groupby('자치구')[['진동_x(g)', '진동_y(g)', '진동_z(g)']].mean().reset_index()

In [9]:
print("\n--- 지역별 평균 소음 수치 ---")
print(noise_by_gu_region.sort_values(by='소음(dB)', ascending=False).head(10))


--- 지역별 평균 소음 수치 ---
      자치구               지역     소음(dB)
85    성동구  roads_and_parks  69.871795
109   은평구      main_street  69.123377
68    마포구  roads_and_parks  67.574194
104  영등포구  roads_and_parks  67.293179
77    서초구      main_street  66.511688
33    광진구  roads_and_parks  66.192308
8     강동구      main_street  65.636857
97    양천구      main_street  65.253247
18    강서구  commercial_area  65.126582
38    구로구  roads_and_parks  64.791423


In [10]:
print("\n--- 자치구별 평균 진동 수치 ---")
print(vibration_by_gu.sort_values(by='진동_x(g)', ascending=False).head(10))


--- 자치구별 평균 진동 수치 ---
     자치구   진동_x(g)   진동_y(g)   진동_z(g)
13  서대문구  0.036502  0.060599  1.039034
22   종로구  0.032561  0.048350  1.043568
12   마포구  0.031706  0.064209  1.043191
1    강동구  0.031403  0.066547  1.038275
0    강남구  0.031071  0.069410  1.036234
6    구로구  0.029871  0.063596  1.050199
5    광진구  0.029559  0.059203  1.042110
10  동대문구  0.028924  0.058839  1.052743
8    노원구  0.028625  0.058502  1.031338
21   은평구  0.028336  0.059949  1.052154


In [13]:
print(f"region_map 컬럼: {region_map.columns.tolist()}")
print(f"df_loc 컬럼: {df_loc.columns.tolist()}")

region_map 컬럼: ['시리얼', '지역']
df_loc 컬럼: ['No', '모델 시리얼(*)', '주소', '좌표 구분코드', '위도', '경도', '변경 전 시리얼', '변경 전 시리얼(데이터 상 표기)', 'Unnamed: 8', '자치구']


In [14]:
import pandas as pd
import glob
import os

# [사전 준비] 매핑 데이터들 컬럼명 정리 및 이름 통일
region_map.columns = region_map.columns.str.strip()
df_loc.columns = df_loc.columns.str.strip()

# df_loc의 시리얼 컬럼명을 '시리얼'로 통일 (가장 중요!)
df_loc = df_loc.rename(columns={'모델 시리얼(*)': '시리얼'})

def process_monthly_data(file_paths, region_map, df_loc):
    all_weekly_data = []

    for path in file_paths:
        try:
            file_name = os.path.basename(path)
            month = file_name.split('.')[1]
            
            # 1. 자동 구분자 감지로 읽기
            df = pd.read_csv(path, encoding='cp949', sep=None, engine='python')
            df.columns = df.columns.str.strip()
            
            # 2. 필요한 컬럼 추출
            cols_to_use = ['시리얼', '소음(dB)', '진동_x(g)', '진동_y(g)', '진동_z(g)']
            existing_cols = [c for c in cols_to_use if c in df.columns]
            
            if '시리얼' not in existing_cols:
                print(f"⚠️ {file_name}: '시리얼' 없음. (건너뜀)")
                continue
                
            df_selected = df[existing_cols].copy()
            
            # 3. 주간 평균 및 월 정보 추가
            weekly = df_selected.groupby('시리얼').mean(numeric_only=True).reset_index()
            weekly['월'] = f"{month}월"
            all_weekly_data.append(weekly)
            
        except Exception as e:
            print(f"❌ 오류 ({file_name}): {e}")

    if not all_weekly_data:
        return None, None
        
    # 4. 전체 데이터 통합
    df_year = pd.concat(all_weekly_data, ignore_index=True)
    df_year.columns = df_year.columns.str.strip()

    # 5. 위치 정보 병합 (이제 '시리얼'로 이름이 같아서 잘 작동합니다)
    df_year = pd.merge(df_year, region_map, on='시리얼', how='left')
    df_year = pd.merge(df_year, df_loc[['시리얼', '자치구']], on='시리얼', how='left')

    # 6. 필터링 및 최종 그룹화
    df_year_filtered = df_year[~df_year['자치구'].isin(['기타(단일주소)', '주소없음'])].dropna(subset=['자치구'])

    # 결과물 생성
    noise_monthly = df_year_filtered.groupby(['월', '자치구', '지역'])['소음(dB)'].mean().reset_index()
    vibration_monthly = df_year_filtered.groupby(['월', '자치구'])[['진동_x(g)', '진동_y(g)', '진동_z(g)']].mean().reset_index()

    return noise_monthly, vibration_monthly

# 실행
noise_monthly, vibration_monthly = process_monthly_data(file_paths, region_map, df_loc)

print("성공적으로 월평균 데이터가 생성되었습니다!")

성공적으로 월평균 데이터가 생성되었습니다!


In [28]:
noise_monthly.to_csv("noise_monthly_2022")

In [27]:
vibration_monthly.to_csv("vibration_monthly_2022")